In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import librosa.display
import logging

from src.helper import *

In [2]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
OUTPUT_PATH = "output/"

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

In [4]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_01_ml_feature_engineering_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting ML feature engineering...")

### Define feature engineering function

In [5]:
def ml_audio_feature_engineering(filepath, logger):
    """
    Extract comprehensive audio features for machine learning.
    """
    features = [
        "duration",
        "sample_rate",
        "rms_energy",
        "zero_crossing_rate",
        "spectral_centroid",
        "spectral_bandwidth",
        "spectral_rolloff",
        "spectral_flatness",
        "chroma_stft_mean",
        "chroma_stft_std",
        "mfcc_1_mean", "mfcc_1_std",
        "mfcc_2_mean", "mfcc_2_std",
        "mfcc_3_mean", "mfcc_3_std",
        "mfcc_4_mean", "mfcc_4_std",
        "mfcc_5_mean", "mfcc_5_std",
    ]

    try:
        # Load audio
        y, sr = librosa.load(filepath, sr=None)  # keep original sample rate
        duration = librosa.get_duration(y=y, sr=sr)

        # Time-domain features
        rms = float(np.mean(librosa.feature.rms(y=y)))
        zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))

        # Spectral features
        centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
        bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
        rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
        flatness = float(np.mean(librosa.feature.spectral_flatness(y=y)))

        # Harmonic/perceptual features
        chroma_stft = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = float(np.mean(chroma_stft))
        chroma_std = float(np.std(chroma_stft))

        # Cepstral features (MFCCs: common in ML)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_features = {}
        for i in range(5):  # just first 5 for compactness
            mfcc_features[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
            mfcc_features[f"mfcc_{i+1}_std"] = float(np.std(mfcc[i]))

        # Final dict
        return {
            "duration": duration,
            "sample_rate": sr,
            "rms_energy": rms,
            "zero_crossing_rate": zcr,
            "spectral_centroid": centroid,
            "spectral_bandwidth": bandwidth,
            "spectral_rolloff": rolloff,
            "spectral_flatness": flatness,
            "chroma_stft_mean": chroma_mean,
            "chroma_stft_std": chroma_std,
            **mfcc_features
        }

    except Exception as e:
        logger.info(f"Error loading {filepath}: {e}")
        return {f: None for f in features}

### Apply feature engineering on train/test/holdout data

In [6]:
records = []

for split in ["holdout", "testing", "training"]:
    for label in ["", "real", "fake"]:
        folder = os.path.join(DATA_PATH, split, label)
        if not os.path.exists(folder):
            continue
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            if os.path.isfile(fpath):
                features = ml_audio_feature_engineering(fpath, logger)
                record = {
                    "filepath": fpath,
                    "filename": fname,
                    "split": split,
                    "label": label,
                }
                record.update(features)
                records.append(record)

df = pd.DataFrame(records)
print(df.shape)
print(df.groupby(["split", "label"]).size())
df.head(3)

/home/ardacandra/miniconda3/envs/deepdetect_audio_deepfake_detection_challenge/lib/python3.13/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
/tmp/ipykernel_46086/2793792379.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filepath, sr=None)  # keep original sample rate
/home/ardacandra/miniconda3/envs/deepdetect_audio_deepfake_detection_challenge/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 50.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
/tmp/ipykernel_46086/2793792379.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(filepath, sr=None)

(103050, 24)
split     label
holdout            14397
testing   fake      4733
          real      6977
training  fake     35212
          real     41731
dtype: int64


,filepath,filename,split,label,duration,sample_rate,rms_energy,zero_crossing_rate,spectral_centroid,spectral_bandwidth,...,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std
0,data/deep-detect/dataset/holdout/audio_06718.wav,audio_06718.wav,holdout,,2.952,16000.0,0.041040,0.165160,1770.421519,1430.811785,...,-358.556580,192.676102,61.244213,68.014946,6.321727,23.635677,17.305662,32.447296,-8.661601,23.757639
1,data/deep-detect/dataset/holdout/audio_00530.wav,audio_00530.wav,holdout,,5.742,16000.0,0.008138,0.127596,1722.722619,1593.515970,...,-464.518829,99.572281,95.552780,48.394844,-0.713542,38.529541,0.020048,27.348408,-4.268337,19.990133
2,data/deep-detect/dataset/holdout/audio_12760.wav,audio_12760.wav,holdout,,1.431,16000.0,0.025449,0.125467,1871.679940,1713.004875,...,-304.998291,98.922905,80.585861,30.346718,-9.771561,27.919638,21.988018,13.912403,-10.379161,21.056404


In [7]:
df.to_csv(os.path.join(OUTPUT_PATH, "nb_01__train_test_holdout_feats.csv"), index=False)
logger.info(f"output features saved to {OUTPUT_PATH}nb_01__train_test_holdout_feats.csv")